In [ ]:
# 1. Imports
import omicverse as ov
import scanpy as sc
import scvi
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import scvelo as scv
import anndata as ad
import pooch
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
import matplotlib as mpl

from matplotlib.colors import LinearSegmentedColormap
import scanpy as sc

# ── 1. Define your extracted color list ────────────────────────────────────────
# (these hex codes correspond approx. to the categories in your legend)


# ── 3. Set figure defaults ────────────────────────────────────────────────────
sc.settings.set_figure_params(dpi=300, facecolor="white")

# 3. Global matplotlib font settings
mpl.rcParams['font.family']      = 'Times New Roman'
mpl.rcParams['font.size']        = 14
mpl.rcParams['axes.titlesize']   = 14
mpl.rcParams['axes.labelsize']   = 14
mpl.rcParams['legend.fontsize']  = 14
mpl.rcParams['xtick.labelsize']  = 14
mpl.rcParams['ytick.labelsize']  = 14

# 4. Scanpy figure params (no fontfamily here)
sc.settings.set_figure_params(
    dpi=300,
    facecolor='white',
    figsize=(6, 6),
    frameon=False,
    fontsize=12
)

# 5. Torch precision
torch.set_float32_matmul_precision("medium")

# 6. Inline backend
%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"

In [ ]:
# Define a custom colormap
custom_cmap = LinearSegmentedColormap.from_list(
    'custom_blue_green', ['cyan', 'red'], N=256
)

In [ ]:
import matplotlib.pyplot as plt

# Make all text bold
plt.rcParams['font.weight']        = 'bold'    # overall font weight :contentReference[oaicite:0]{index=0}
plt.rcParams['axes.titleweight']   = 'bold'    # axis titles :contentReference[oaicite:1]{index=1}
plt.rcParams['axes.labelweight']   = 'bold'    # axis labels :contentReference[oaicite:2]{index=2}
#plt.rcParams['legend.fontweight']  = 'bold'    # legend text :contentReference[oaicite:3]{index=3}
plt.rcParams['xtick.color']        = 'black'   # ensure ticks remain visible
plt.rcParams['ytick.color']        = 'black'

In [ ]:
import os
# 1. Change to your desired path
os.chdir("D:/AI work/scRNA_ad")
# 2. Verify it took effect
print("Current working dir:", os.getcwd())

In [ ]:
adata = sc.read_h5ad('./zhang/adata_microglia_20250519.h5ad')

In [ ]:
adata = sc.read_h5ad('./zhang/adata_without_lowquality_20250519.h5ad')

In [ ]:
adata

In [ ]:
# Load meta data
meta_path = './zhang/final/meta_final.csv'
meta = pd.read_csv(meta_path, index_col=0)

# Load UMAP coordinates
umap_path = './zhang/final/umap_final.csv'
umap_df = pd.read_csv(umap_path, index_col=0)

# If you already have an AnnData object, assign obs and obsm
adata.obs = meta
adata.obsm['X_umap'] = umap_df.values
# Load metadata
#meta_path = './zhang/final/meta.csv'
#meta = pd.read_csv(meta_path, index_col=0)

# Convert only the 'RNA_snn_res.3' column to string
#target_col = 'RNA_snn_res.3'
#if target_col in meta.columns:
 #   meta[target_col] = meta[target_col].astype(str)
#else:
 #   raise KeyError(f"Column '{target_col}' not found in metadata.")

# Check
print(adata)
print("Meta data columns:", adata.obs.columns)
print("UMAP shape:", adata.obsm['X_umap'].shape)

In [ ]:
adata

In [ ]:
import omicverse as ov

In [ ]:
ov.pl.embedding(adata,
                basis='X_umap',
                color='class',
                frameon='small')#,
                #legend_loc='on data')

In [ ]:
ov.pl.embedding(adata,
                basis='X_umap',
                color='sample',
                frameon='small')#,
                #legend_loc='on data')

In [ ]:
# 定义需要排除的 subtype
excluded = ['lowQ', 'Unknown']

# 对 AnnData 对象进行子集化，排除掉 obs 中 subtype 为 'lowQ' 或 'Unknown' 的细胞
adata = adata[~adata.obs['subtype'].isin(excluded)].copy()

# 检查排除后的 subtype 信息
print("Remaining subtypes:", adata.obs['subtype'].unique())

In [ ]:
adata

In [ ]:
# Ensure 'RNA_snn_res.3' is treated as categorical
adata.obs['RNA_snn_res.3'] = adata.obs['RNA_snn_res.3'].astype('category')

In [ ]:
adata

In [ ]:
ov.pl.embedding(adata,
                basis='X_umap',
                color='class',
                frameon='small')

In [ ]:
marker_genes = ['CX3CR1','SORL1']
sc.pl.dotplot(adata, marker_genes, groupby=['sample'],
             standard_scale='var');

In [ ]:
adata

In [ ]:
import scanpy as sc
import pandas as pd

# Convert cluster labels to strings.
adata.obs['RNA_snn_res.3'] = adata.obs['RNA_snn_res.3'].astype(str)

# Check how many cells are in each cluster.
group_counts = adata.obs['RNA_snn_res.3'].value_counts()
print("Cell counts per group:")
print(group_counts)

# Filter out clusters with only one cell.
valid_groups = group_counts[group_counts > 1].index.tolist()
print("Clusters with >1 cell:", valid_groups)

# Subset the AnnData object to only include valid groups.
adata_sub = adata[adata.obs['RNA_snn_res.3'].isin(valid_groups)].copy()

# Run differential expression analysis using the 'wilcoxon' method.
sc.tl.rank_genes_groups(adata_sub, groupby='RNA_snn_res.3', method='wilcoxon')

# Optional: Inspect the top 5 marker genes per cluster.
sc.pl.rank_genes_groups(adata_sub, n_genes=5, sharey=False)

# Visualize the top markers using the built-in dotplot function.
sc.pl.rank_genes_groups_dotplot(
    adata_sub,
    n_genes=5,
    groupby='RNA_snn_res.3',
    standard_scale='var'
)

In [ ]:
adata.write_h5ad('./zhang/humanAD_filtered20250518_2.h5ad')

In [ ]:
adata

In [ ]:
# Boolean mask where class equals "Microglia"
mask = adata.obs['class'] == 'Microglia'

# Subset and make a copy to avoid view warnings
adata_microglia = adata[mask, :].copy()

# Inspect
print(adata_microglia)

In [ ]:
adata_microglia

In [ ]:
import scanpy as sc
import pandas as pd

# Convert cluster labels to strings.
adata_microglia.obs['RNA_snn_res.3'] = adata_microglia.obs['RNA_snn_res.3'].astype(str)

# Check how many cells are in each cluster.
group_counts = adata_microglia.obs['RNA_snn_res.3'].value_counts()
print("Cell counts per group:")
print(group_counts)

# Filter out clusters with only one cell.
valid_groups = group_counts[group_counts > 5].index.tolist()
print("Clusters with >1 cell:", valid_groups)

# Subset the AnnData object to only include valid groups.
adata_microglia_sub = adata_microglia[adata_microglia.obs['RNA_snn_res.3'].isin(valid_groups)].copy()

# Run differential expression analysis using the 'wilcoxon' method.
sc.tl.rank_genes_groups(adata_microglia_sub, groupby='RNA_snn_res.3', method='wilcoxon')

# Optional: Inspect the top 5 marker genes per cluster.
sc.pl.rank_genes_groups(adata_microglia_sub, n_genes=10, sharey=False)

# Visualize the top markers using the built-in dotplot function.
sc.pl.rank_genes_groups_dotplot(
    adata_microglia_sub,
    n_genes=10,
    groupby='RNA_snn_res.3',
    standard_scale='var'
)

In [ ]:
adata

In [ ]:
adata

In [ ]:
group_map = {
    'Low_3_2':  'AD',      # Previously Low_Tau
    'Low_2_2':  'Non_AD',
    'Low_5_2':  'AD',      # Previously Low_Tau
    'Low_4_2':  'AD',      # Previously Low_Tau
    'Low_12_2': 'AD',      # Previously Low_Tau
    'Low_13_2': 'Non_AD',
    'Low_11_2': 'AD',      # Previously Low_Tau
    'Low_10_2': 'AD',      # Previously Low_Tau
    'Low_16_2': 'AD',      # Previously Low_Tau
    'Low_31':   'AD',      # Previously Low_Tau
    'High_7_2':  'AD',     # Previously High_Tau
    'High_1_2':  'AD',     # Previously High_Tau
    'High_9_2':  'AD',     # Previously High_Tau
    'High_19_2': 'AD',     # Previously High_Tau
    'High_33':   'AD',     # Previously High_Tau
    'High_34':   'AD',     # Previously High_Tau
    'Non_1':    'Non_AD',
    'Non_28':   'AD',      # Previously High_Tau
    'Non_2':    'Non_AD',
    'Non_35':   'Non_AD',
    'Non_8_2':  'Non_AD',
    'Non_14_2': 'Non_AD'
}
adata.obs['AD'] = adata.obs['sample'].map(group_map).astype('category')

In [ ]:
# 1. Map samples to broader groups
group_map = {
    'Low_3_2':  'Low(3-2)',
    'Low_2_2':  'Non(2-2)',
    'Low_5_2':  'Low(5-2)',
    'Low_4_2':  'Low(4-2)',
    'Low_12_2': 'Low(12-2)',
    'Low_13_2': 'Non(13-2)',
    'Low_11_2': 'Low(11-2)',
    'Low_10_2': 'Low(10-2)',
    'Low_16_2': 'Low(16-2)',
    'Low_31':   'Low(31)',
    'High_7_2':  'High(7-2)',
    'High_1_2':  'High(1-2)',
    'High_9_2':  'High(9-2)',
    'High_19_2': 'High(19-2)',
    'High_33':   'High(33)',
    'High_34':   'High(34)',
    'Non_1':    'Non(1)',
    'Non_28':   'High(28)',
    'Non_2':    'Non(2)',
    'Non_35':   'Non(35)',
    'Non_8_2':  'Non(8-2)',
    'Non_14_2': 'Non(14-2)'
}

adata.obs['batch'] = adata.obs['sample'].map(group_map).astype('category')

In [ ]:
# 1. Map samples to broader groups
group_map = {
    'Low_3_2':  '76',
    'Low_2_2':  '76',
    'Low_5_2':  '92',
    'Low_4_2':  '82',
    'Low_12_2': '67',
    'Low_13_2': '71',
    'Low_11_2': '77',
    'Low_10_2': '62',
    'Low_16_2': '71',
    'Low_31':   '75',
    'High_7_2':  '92',
    'High_1_2':  '95',
    'High_9_2':  '89',
    'High_19_2': '84',
    'High_33':   '81',
    'High_34':   '93',
    'Non_1':    '95',
    'Non_28':   '71',
    'Non_2':    '62',
    'Non_35':   '61',
    'Non_8_2':  '68',
    'Non_14_2': '73'
}

adata.obs['age'] = adata.obs['sample'].map(group_map).astype('category')

In [ ]:
marker_genes = ['CX3CR1','GJA1','SLC17A7','SORL1']
sc.pl.dotplot(adata, marker_genes, groupby=['class'],
             standard_scale='var');

In [ ]:
adata

In [ ]:
adata.write_h5ad('./zhang/final_all_20250519.h5ad')

In [ ]:
# Create a boolean mask for cells of interest
mask = adata.obs['class'].isin(['Astro'])

# Subset the AnnData to only those cells
adata_micro = adata[mask].copy()

In [ ]:
adata_micro

In [ ]:
ov.pl.embedding(adata_micro,
                basis='X_umap',
                color='RNA_snn_res.3',
                frameon='small')

In [ ]:
# Preprocess
adata_micro = ov.pp.preprocess(
    adata_micro,
    mode='shiftlog|pearson',
    n_HVGs=3000,
)

# Store raw
adata_micro.raw = adata_micro

# Subset to highly variable genes
adata_micro = adata_micro[:, adata_micro.var.highly_variable_features]

# Scale
ov.pp.scale(adata_micro)

# PCA
ov.pp.pca(
    adata_micro,
    layer='scaled',
    n_pcs=50
)

# Compute neighborhood graph
ov.pp.neighbors(
    adata_micro,
    use_rep='scaled|original|X_pca',
    n_neighbors=15,
    n_pcs=30
)

# UMAP embedding
ov.pp.umap(
    adata_micro,
    min_dist=1
)

# Leiden clustering
ov.pp.leiden(
    adata_micro,
    resolution=0.5
)
# Plot UMAP
ov.pl.embedding(
    adata_micro,
    basis='X_umap',
    color=['leiden', 'AD'],
    frameon='small',
    cmap='Reds',
    legend_loc='on data'
)

In [ ]:
sc.tl.dendrogram(adata_micro,'leiden',use_rep='scaled|original|X_pca')
sc.tl.rank_genes_groups(adata_micro, 'leiden', use_rep='scaled|original|X_pca',
                        method='t-test',use_raw=False,key_added='leiden_ttest')
sc.pl.rank_genes_groups_dotplot(adata_micro,groupby='leiden',
                                cmap='Spectral_r',key='leiden_ttest',
                                standard_scale='var',n_genes=5)

In [ ]:
import scanpy as sc
import pandas as pd

# 1. Run differential expression to find markers for cluster 0
#    This will compare cells in leiden=='0' against all other cells.
sc.tl.rank_genes_groups(
    adata_micro,
    groupby='leiden',
    groups=['8'],
    reference='rest',        # compare against all other clusters
    method='t-test_overestim_var',  # you can also try 'wilcoxon', 'logreg', etc.
    n_genes=adata_micro.shape[0]    # rank all genes; or set to top N
)

# 2. Extract results into a DataFrame
markers_8 = sc.get.rank_genes_groups_df(adata_micro, group='8')

# 3. Inspect top markers
print(markers_8.head(10))

# 4. (Optional) Save to CSV
markers_8.to_csv("adata_micro_leiden1_markers.csv", index=False)

In [ ]:
marker_genes = ['SLC1A2','ARHGAP24','SLC1A3']
sc.pl.dotplot(adata_micro, marker_genes, groupby='leiden',standard_scale='var',figsize=(4,4));

In [ ]:
group_map = {
    '0':  'Inflammation(MEG3)',      # Previously Low_Tau
    '1':  'ECM(ABI3BP)',
    '2':  'Mitophagy(RPL13)',
    '3': 'Homeostatic(ARHGAP24)',
    '4': 'Synaptic(HIF3A)',
    '5': 'Reactive(GFAP)',
    '6': 'Abeta modulation(HPSE2)',
    '7': 'Reactive(DPP10)',
    '8': 'Inflammation(ST18)',
    '9': 'Lipid(APOE)',
    '10': 'Homeostatic(SLC1A2)',
    '11': 'Reactive2(DPP10)',
    '12': 'Inteferon(IFIT1)'
}
adata_micro.obs['astrocyte_subtype'] = adata_micro.obs['leiden'].map(group_map).astype('category')

In [ ]:
# Plot UMAP
ov.pl.embedding(
    adata_micro,
    basis='X_umap',
    color=['astrocyte_subtype', 'AD'],
    frameon='small',
    cmap='Reds',
    legend_loc='on data'
)

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize = (3,6))
ov.pl.cellproportion(adata=adata_micro,celltype_clusters='astrocyte_subtype',
                    groupby='AD',legend=True,ax=ax)

In [ ]:
adata_micro

In [ ]:
# Plot UMAP
ov.pl.embedding(
    adata_micro,
    basis='X_umap',
    color=['leiden', 'AD'],
    frameon='small',
    cmap='Reds',
    legend_loc='on data'
)

In [ ]:
adata

In [ ]:
# Define the list of obs-columns you want
meta_cols = [
    'sample',
    'class',
    'subtype',
    'RNA_snn_res.3',
    'Tau',
    'AD',
    'batch',
    'age',
    'leiden',
    'astrocyte_subtype'
]

# 1) Extract into a standalone DataFrame
metadata_df = adata_micro.obs[meta_cols].copy()

# (Optional) Inspect
print(metadata_df.head())

# 2) Save to disk
metadata_df.to_csv("./zhang/adata_metadata_microglia.csv", index=True)
# This creates adata_micro_metadata.csv with cell barcodes as the index

In [ ]:
adata_micro.write_h5ad('./zhang/adata_astrocyte_20250519.h5ad')

In [ ]:
marker_genes = ['GJA1','SLC17A7','GAD1','CX3CR1','PDGFRA','MOG','CDH5','SORL1']

In [ ]:
marker_genes = ['SORL1']

In [ ]:
adata

In [ ]:
import scanpy as sc

# 1. Create the DotPlot object
dp = sc.pl.dotplot(
    adata,
    marker_genes,
    groupby=['class', 'Tau'],
    standard_scale='var',
    cmap=custom_cmap,
    return_fig=True,  # returns a DotPlot object
    swap_axes=True
)

# 2. Build the figure explicitly
dp.make_figure()

# 3. Remove top and right spines from all axes
for ax in dp.fig.axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# 4. Save the figure without displaying it
dp.fig.savefig("./output/dotplot_output_tau_20250609.tif", dpi=300, format='tiff', bbox_inches='tight')

In [ ]:
adata.write_h5ad('./zhang/humanAD_filitered20250518.h5ad')

In [ ]:
# 1. Map samples to broader groups
group_map = {
    'Low_3_2':  'Low_Tau',
    'Low_2_2':  'Non_AD',
    'Low_5_2':  'Low_Tau',
    'Low_4_2':  'Low_Tau',
    'Low_12_2': 'Low_Tau',
    'Low_13_2': 'Non_AD',
    'Low_11_2': 'Low_Tau',
    'Low_10_2': 'Low_Tau',
    'Low_16_2': 'Low_Tau',
    'Low_31':   'Low_Tau',
    'High_7_2':  'High_Tau',
    'High_1_2':  'High_Tau',
    'High_9_2':  'High_Tau',
    'High_19_2': 'High_Tau',
    'High_33':   'High_Tau',
    'High_34':   'High_Tau',
    'Non_1':    'Non_AD',
    'Non_28':   'High_Tau',
    'Non_2':    'Non_AD',
    'Non_35':   'Non_AD',
    'Non_8_2':  'Non_AD',
    'Non_14_2': 'Non_AD'
}

adata.obs['Tau'] = adata.obs['sample'].map(group_map).astype('category')

In [ ]:
adata

In [ ]:
# 1. Map samples to broader groups
group_map = {
    'Low_3_2':  'Low_Tau',
    'Low_2_2':  'Non_AD',
    'Low_5_2':  'Low_Tau',
    'Low_4_2':  'Low_Tau',
    'Low_12_2': 'Low_Tau',
    'Low_13_2': 'Non_AD',
    'Low_11_2': 'Low_Tau',
    'Low_10_2': 'Low_Tau',
    'Low_16_2': 'Low_Tau',
    'Low_31':   'Low_Tau',
    'High_7_2':  'High_Tau',
    'High_1_2':  'High_Tau',
    'High_9_2':  'High_Tau',
    'High_19_2': 'High_Tau',
    'High_33':   'High_Tau',
    'High_34':   'High_Tau',
    'Non_1':    'Non_AD',
    'Non_28':   'High_Tau',
    'Non_2':    'Non_AD',
    'Non_35':   'Non_AD',
    'Non_8_2':  'Non_AD',
    'Non_14_2': 'Non_AD'
}

adata.obs['AD'] = adata.obs['sample'].map(group_map).astype('category')

In [ ]:
adata

In [ ]:
group_map = {
    'Low_3_2':  'AD',      # Previously Low_Tau
    'Low_2_2':  'Non_AD',
    'Low_5_2':  'AD',      # Previously Low_Tau
    'Low_4_2':  'AD',      # Previously Low_Tau
    'Low_12_2': 'AD',      # Previously Low_Tau
    'Low_13_2': 'Non_AD',
    'Low_11_2': 'AD',      # Previously Low_Tau
    'Low_10_2': 'AD',      # Previously Low_Tau
    'Low_16_2': 'AD',      # Previously Low_Tau
    'Low_31':   'AD',      # Previously Low_Tau
    'High_7_2':  'AD',     # Previously High_Tau
    'High_1_2':  'AD',     # Previously High_Tau
    'High_9_2':  'AD',     # Previously High_Tau
    'High_19_2': 'AD',     # Previously High_Tau
    'High_33':   'AD',     # Previously High_Tau
    'High_34':   'AD',     # Previously High_Tau
    'Non_1':    'Non_AD',
    'Non_28':   'AD',      # Previously High_Tau
    'Non_2':    'Non_AD',
    'Non_35':   'Non_AD',
    'Non_8_2':  'Non_AD',
    'Non_14_2': 'Non_AD'
}
adata.obs['AD'] = adata.obs['sample'].map(group_map).astype('category')

In [ ]:
import pandas as pd

# Define the desired order
desired_order = ['Non_AD', 'Low_Tau', 'High_Tau']

# Ensure 'Tau' is a categorical variable with the specified order
adata.obs['Tau'] = pd.Categorical(adata.obs['Tau'], categories=desired_order, ordered=True)

# Define the desired order
desired_order = ['Non_AD', 'AD']

# Ensure 'Tau' is a categorical variable with the specified order
adata.obs['AD'] = pd.Categorical(adata.obs['AD'], categories=desired_order, ordered=True)

In [ ]:
# 1. Create the DotPlot object
dp = sc.pl.dotplot(
    adata,
    marker_genes,
    groupby=['class','AD'],
    standard_scale='var',
    cmap=custom_cmap,
    swap_axes=True,
    figsize=(12,6),
    color_map=custom_cmap,
    return_fig=True  # returns a DotPlot, not a Figure
)

# 2. Get the axes dict
axes_dict = dp.get_axes()  # returns {'mainplot_ax', 'size_legend_ax', 'color_legend_ax'} :contentReference[oaicite:0]{index=0}

# 3. Select the main plotting axes
ax = axes_dict['mainplot_ax']

# 4. Remove the top & right spines entirely
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# 6. (Optional) Tweak the tick marks to match
ax.tick_params(direction='in', length=4, width=thin_width)

# 7. Render the updated plot
dp.show()

In [ ]:
adata

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os

# The AnnData object is already loaded
# We'll use the existing 'adata' object that's already in memory

# Verify that the required columns exist
required_columns = ['class', 'AD']
for col in required_columns:
    if col not in adata.obs.columns:
        raise ValueError(f"Required column '{col}' not found in adata.obs")

# Verify that 'AD' column contains both 'AD' and 'Non_AD' values
ad_categories = adata.obs['AD'].unique()
if 'AD' not in ad_categories or 'Non_AD' not in ad_categories:
    print(f"Warning: 'AD' column should contain both 'AD' and 'Non_AD' values. Found: {ad_categories}")

# Print dataset info
print(f"Dataset has {adata.n_obs} cells and {adata.n_vars} genes")
print(f"AD samples: {np.sum(adata.obs['AD'] == 'AD')}")
print(f"Non_AD samples: {np.sum(adata.obs['AD'] == 'Non_AD')}")

# Make sure the expression matrix is log-transformed
if "log1p" not in adata.uns.get("preprocessing", {}):
    print("Normalizing and log-transforming data...")
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    print("Data normalized and log-transformed")
else:
    print("Data already log-transformed")

# Get unique cell types
cell_types = adata.obs['class'].unique()
print(f"Found {len(cell_types)} cell types: {', '.join(cell_types)}")

# Create directory for results if it doesn't exist
results_dir = "DE_results_new"
os.makedirs(results_dir, exist_ok=True)

# Process each cell type
for cell_type in cell_types:
    print(f"\nProcessing cell type: {cell_type}")
    
    # Subset data for the current cell type
    adata_subset = adata[adata.obs['class'] == cell_type].copy()
    
    # Check if we have both AD and Non_AD samples in this cell type
    ad_count = np.sum(adata_subset.obs['AD'] == 'AD')
    non_ad_count = np.sum(adata_subset.obs['AD'] == 'Non_AD')
    print(f"  AD samples: {ad_count}, Non_AD samples: {non_ad_count}")
    
    if ad_count == 0 or non_ad_count == 0:
        print(f"  Warning: One group has 0 samples. Skipping.")
        continue
    
    # Perform rank_genes_groups using Wilcoxon test (Non_AD as control/reference)
    try:
        sc.tl.rank_genes_groups(
            adata_subset, 
            groupby='AD', 
            reference='Non_AD',
            groups=['AD'], 
            method='wilcoxon',
            key_added=f"rank_genes_{cell_type}"
        )
        
        # Extract results into a DataFrame
        de_results = sc.get.rank_genes_groups_df(
            adata_subset, 
            group='AD', 
            key=f"rank_genes_{cell_type}"
        )
        
        # Add cell type information
        de_results['cell_type'] = cell_type
        
        # Save to CSV
        output_file = os.path.join(results_dir, f"DE_genes_{cell_type}_AD_vs_NonAD.csv")
        de_results.to_csv(output_file, index=False)
        print(f"  Results saved to {output_file}")
        
        # Display top 10 DE genes
        print(f"  Top 10 differentially expressed genes:")
        print(de_results.head(10)[['names', 'scores', 'pvals', 'pvals_adj', 'logfoldchanges']])
    
    except Exception as e:
        print(f"  Error processing cell type {cell_type}: {e}")

print("\nDifferential expression analysis complete.")
print(f"Results saved in directory: {results_dir}")

# Combine all results into a single file
all_results = []
for cell_type in cell_types:
    file_path = os.path.join(results_dir, f"DE_genes_{cell_type}_AD_vs_NonAD.csv")
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        all_results.append(df)

if all_results:
    combined_results = pd.concat(all_results, axis=0)
    combined_file = os.path.join(results_dir, "DE_genes_all_celltypes.csv")
    combined_results.to_csv(combined_file, index=False)
    print(f"\nCombined results saved to: {combined_file}")

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import sys

# Load the AnnData object if not already loaded
# Uncomment the line below and specify your data file if needed
# adata = sc.read_h5ad("your_data_file.h5ad")

# Check if adata is defined
try:
    adata
    print("Using existing AnnData object")
except NameError:
    print("Error: No AnnData object named 'adata' found.")
    print("Please load your data before running this script.")
    sys.exit(1)

# Verify that the required columns exist
required_columns = ['class', 'AD']
for col in required_columns:
    if col not in adata.obs.columns:
        raise ValueError(f"Required column '{col}' not found in adata.obs")

# Verify that 'AD' column contains both 'AD' and 'Non_AD' values
ad_categories = adata.obs['AD'].unique()
if 'AD' not in ad_categories or 'Non_AD' not in ad_categories:
    print(f"Warning: 'AD' column should contain both 'AD' and 'Non_AD' values. Found: {ad_categories}")

# Print dataset info
print(f"Dataset has {adata.n_obs} cells and {adata.n_vars} genes")
print(f"AD samples: {np.sum(adata.obs['AD'] == 'AD')}")
print(f"Non_AD samples: {np.sum(adata.obs['AD'] == 'Non_AD')}")

# Make sure the expression matrix is log-transformed
if "log1p" not in adata.uns.get("preprocessing", {}):
    print("Normalizing and log-transforming data...")
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    print("Data normalized and log-transformed")
else:
    print("Data already log-transformed")

# Get unique cell types
cell_types = adata.obs['class'].unique()
print(f"Found {len(cell_types)} cell types: {', '.join(cell_types)}")

# Create directory for results if it doesn't exist
results_dir = "DE_results_new"
os.makedirs(results_dir, exist_ok=True)

# Process each cell type
for cell_type in cell_types:
    print(f"\nProcessing cell type: {cell_type}")
    
    # Subset data for the current cell type
    adata_subset = adata[adata.obs['class'] == cell_type].copy()
    
    # Check if we have both AD and Non_AD samples in this cell type
    ad_count = np.sum(adata_subset.obs['AD'] == 'AD')
    non_ad_count = np.sum(adata_subset.obs['AD'] == 'Non_AD')
    print(f"  AD samples: {ad_count}, Non_AD samples: {non_ad_count}")
    
    if ad_count == 0 or non_ad_count == 0:
        print(f"  Warning: One group has 0 samples. Skipping.")
        continue
    
    # Perform rank_genes_groups using t-test (Non_AD as control/reference)
    try:
        sc.tl.rank_genes_groups(
            adata_subset, 
            groupby='AD',
            reference='Non_AD',
            groups=['AD'], 
            method='t-test',  # Changed from 'wilcoxon' to 't-test'
            key_added=f"rank_genes_{cell_type}"
        )
        
        # Extract results into a DataFrame
        de_results = sc.get.rank_genes_groups_df(
            adata_subset, 
            group='AD',
            key=f"rank_genes_{cell_type}"
        )
        
        # Add cell type information
        de_results['cell_type'] = cell_type
        
        # Save to CSV
        output_file = os.path.join(results_dir, f"DE_genes_{cell_type}_AD_vs_NonAD.csv")
        de_results.to_csv(output_file, index=False)
        print(f"  Results saved to {output_file}")
        
        # Display top 10 DE genes
        print(f"  Top 10 differentially expressed genes:")
        print(de_results.head(10)[['names', 'scores', 'pvals', 'pvals_adj', 'logfoldchanges']])
        
    except Exception as e:
        print(f"  Error processing cell type {cell_type}: {e}")

print("\nDifferential expression analysis complete.")
print(f"Results saved in directory: {results_dir}")

# Combine all results into a single file
all_results = []
for cell_type in cell_types:
    file_path = os.path.join(results_dir, f"DE_genes_{cell_type}_AD_vs_NonAD.csv")
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        all_results.append(df)

if all_results:
    combined_results = pd.concat(all_results, axis=0)
    combined_file = os.path.join(results_dir, "DE_genes_all_celltypes.csv")
    combined_results.to_csv(combined_file, index=False)
    print(f"\nCombined results saved to: {combined_file}")
else:
    print("\nNo results were generated, so no combined file was created.")

In [ ]:
ov.pl.embedding(adata,
                basis='X_umap',
                color='class',
                frameon='small')

In [ ]:
sc.pl.umap(
    adata,
    color=['class','AD','Tau'],
    cmap=custom_cmap,
    size=15,
    alpha=0.8,
    frameon=False,
    #legend_loc = 'on data',
    show=True
)

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize = (2,6))
ov.pl.cellproportion(adata=adata,celltype_clusters='class',
                    groupby='AD',legend=True,ax=ax)

In [ ]:
adata.write_h5ad('./zhang/adata_without_lowquality_20250519.h5ad')

In [ ]:
adata = sc.read_h5ad('./zhang/adata_without_lowquality_20250519.h5ad')

In [ ]:
adata

In [ ]:
adata

In [ ]:
# 2. microglia subset of data subset data analysis

In [ ]:
adata

In [ ]:
adata=sc.read_h5ad('D:/AI work/scRNA_ad/zhang/adata_microglia_20250519.h5ad')

In [ ]:
adata

In [ ]:
color_list_random_13 = [
    '#4CFFFF', '#CC3333', '#009999', '#3333CC', '#FF0000',
    '#00CCCC', '#000099', '#4C4CFF', '#CC0000', '#00FFFF',
    '#990000', '#0000FF', '#FF4C4C'
]

In [ ]:
adata

In [ ]:
sc.tl.pca(adata,n_comps=50)
sc.pp.neighbors(adata,n_neighbors=15,use_rep='X_mde_harmony')
sc.tl.umap(adata)

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc

# Determine how many subtypes you have
unique_subtypes = adata.obs['microglia_subtype'].unique()
n = len(unique_subtypes)

# Choose your palette (tab20 up to 20 colors)
palette = plt.get_cmap('tab20').colors[:n]

# 1. Create a Figure with the desired size and DPI
#fig, ax = plt.subplots(dpi=300)

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc

# Assuming `palette` is your list/array of distinct colors:
# e.g. palette = plt.get_cmap('tab20').colors[:n]
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize = (4,4))
sc.pl.umap(
    adata,
    color='microglia_subtype',   # or ['microglia_subtype']
    palette=palette,             # categorical palette
    size=20,
    alpha=0.8,
    frameon=False,
    #legend_loc='on data',        # place labels on the points
    show=True
)

In [ ]:
import pandas as pd

# Define the desired order
desired_order = ['Non_AD', 'Low_Tau', 'High_Tau']

# Ensure 'Tau' is a categorical variable with the specified order
adata.obs['Tau'] = pd.Categorical(adata.obs['Tau'], categories=desired_order, ordered=True)


# Define the desired order
desired_order = ['Non_AD', 'AD']

# Ensure 'Tau' is a categorical variable with the specified order
adata.obs['AD'] = pd.Categorical(adata.obs['AD'], categories=desired_order, ordered=True)

In [ ]:
import pandas as pd

# Define the desired order
desired_order = ['Exn','InN','Astro','OPC','Oligo','Micro/immune','Vas']

# Ensure 'Tau' is a categorical variable with the specified order
adata.obs['class'] = pd.Categorical(adata.obs['class'], categories=desired_order, ordered=True)

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize = (2,4))
ov.pl.cellproportion(adata=adata,celltype_clusters='microglia_subtype',
                    groupby='Tau',legend=True,ax=ax)

In [ ]:
adata

In [ ]:
marker_genes=['SLC17A7','GAD1','GJA1','PDGFRA','MOG','CX3CR1','PECAM1','SORL1']

In [ ]:
sc.pl.dotplot(adata, marker_genes, groupby='class',
             standard_scale='var')

In [ ]:
print(marker_genes)
print(type(marker_genes))

# If it's a list
if isinstance(marker_genes, list):
    for i, g in enumerate(marker_genes):
        if not isinstance(g, str):
            print(f"Non-string gene at index {i}: {g} ({type(g)})")

# If it's a dict (Scanpy also supports dicts of marker genes for grouped plotting)
elif isinstance(marker_genes, dict):
    for group, genes in marker_genes.items():
        for i, g in enumerate(genes):
            if not isinstance(g, str):
                print(f"Non-string gene in group '{group}' at index {i}: {g} ({type(g)})")


In [ ]:
adata

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

# 1. Create the DotPlot object
dp = sc.pl.dotplot(
    adata,
    marker_genes,
    groupby='group',
    standard_scale='var',
    cmap=custom_cmap,
    return_fig=True,
    swap_axes=True
)

# 2. Get the axes dictionary
axes_dict = dp.get_axes()

# 3. Select the main plotting axes
ax = axes_dict['mainplot_ax']

# 4. Remove the top & right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# 5. Thin the bottom & left spines
thin_width = 0.5
ax.spines['bottom'].set_linewidth(thin_width)
ax.spines['left'].set_linewidth(thin_width)

# 6. Adjust tick marks
ax.tick_params(direction='in', length=4, width=thin_width)

# 7. Set figure size before rendering
#plt.gcf().set_size_inches(4, 6)

# 8. Render the plot (displays and sets current figure)
dp.show()

# 9. Get the current figure and save as TIFF
fig = plt.gcf()
output_path = "./output/dotplot-20250527-final.tif"
fig.savefig(output_path, dpi=300, format='tiff', bbox_inches='tight', pil_kwargs={'compression': 'tiff_lzw'})

In [ ]:
dp = sc.pl.dotplot(
    adata,
    marker_genes,
    groupby=['class','Tau'],
    standard_scale='var',
    cmap=custom_cmap,
    return_fig=True,
    swap_axes=True
)

In [ ]:
# 1. Create the DotPlot object
dp = sc.pl.dotplot(
    adata,
    marker_genes,
    groupby=['class','Tau'],
    standard_scale='var',
    cmap=custom_cmap,
    return_fig=True,  # returns a DotPlot, not a Figure
    swap_axes= True
)

# 2. Get the axes dict
axes_dict = dp.get_axes()  # returns {'mainplot_ax', 'size_legend_ax', 'color_legend_ax'} :contentReference[oaicite:0]{index=0}

# 3. Select the main plotting axes
ax = axes_dict['mainplot_ax']

# 4. Remove the top & right spines entirely
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# 5. Thin the bottom & left spines (e.g. half the default width)
thin_width = 0.5
ax.spines['bottom'].set_linewidth(thin_width)
ax.spines['left'].set_linewidth(thin_width)

# 6. (Optional) Tweak the tick marks to match
ax.tick_params(direction='in', length=4, width=thin_width)

# 7. Render the updated plot
dp.show()

In [ ]:
# 1. Define your group map with real numbers, not strings
group_map = {
    'Low_3_2':   0.44,
    'Low_2_2':   0.395,
    'Low_5_2':   0.615,
    'Low_4_2':   0.145,
    'Low_12_2':  0.118,
    'Low_13_2':  0.095,
    'Low_11_2':  0.485,
    'Low_10_2':  0.241,
    'Low_16_2':  0.506,
    'Low_31':    0.418,
    'High_7_2':  0.957,
    'High_1_2':  0.262,
    'High_9_2':  0.284,
    'High_19_2': 1.538,
    'High_33':   1.557,
    'High_34':   1.522,
    'Non_1':     0.262,
    'Non_28':    0.572,
    'Non_2':     0.014,
    'Non_35':    0.026,
    'Non_8_2':   0.032,
    'Non_14_2':  0.032
}

# 2. Map and convert to float
adata_micro.obs['tau_number'] = (
    adata_micro.obs['sample']
    .map(group_map)        # this yields floats or NaN
    .astype(float)         # ensure the dtype is numeric
)

# 3. (Optionally) check dtype and missing values
print(adata_micro.obs['tau_number'].dtype)  # should be float64
print(adata_micro.obs['tau_number'].isna().sum(), "missing mappings")

In [ ]:
# 1. Define your group map with real numbers, not strings
group_map = {
    'Low_3_2':   0.44,
    'Low_2_2':   0.395,
    'Low_5_2':   0.615,
    'Low_4_2':   0.145,
    'Low_12_2':  0.118,
    'Low_13_2':  0.095,
    'Low_11_2':  0.485,
    'Low_10_2':  0.241,
    'Low_16_2':  0.506,
    'Low_31':    0.418,
    'High_7_2':  0.957,
    'High_1_2':  0.262,
    'High_9_2':  0.284,
    'High_19_2': 1.538,
    'High_33':   1.557,
    'High_34':   1.522,
    'Non_1':     0.262,
    'Non_28':    0.572,
    'Non_2':     0.014,
    'Non_35':    0.026,
    'Non_8_2':   0.032,
    'Non_14_2':  0.032
}

# 2. Map and convert to float
adata_micro.obs['tau_number'] = (
    adata_micro.obs['sample']
    .map(group_map)        # this yields floats or NaN
    .astype(float)         # ensure the dtype is numeric
)

# 3. (Optionally) check dtype and missing values
print(adata_micro.obs['tau_number'].dtype)  # should be float64
print(adata_micro.obs['tau_number'].isna().sum(), "missing mappings")

In [ ]:
ov.pl.embedding(adata_micro,basis='X_umap',
                   color=['tau_number'],
                   frameon='small',cmap='Reds')

In [ ]:
adata_micro

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Extract the obs DataFrame
obs = adata_micro.obs.copy()

# 2. Compute mean tau_number per micro_subtype
mean_tau = (
    obs
    .groupby('microglia_subtype')['tau_number']
    .mean()
    .reset_index(name='mean_tau')
)

# 3. Print the table
print(mean_tau)

# 4. Plot as a bar chart
plt.figure(figsize=(8, 4))
plt.bar(mean_tau['microglia_subtype'], mean_tau['mean_tau'])
plt.xticks(rotation=45, ha='right')
plt.ylabel('Average τ (tau_number)')
plt.title('Mean τ per Micro Subtype')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Extract the obs DataFrame
obs = adata_micro.obs.copy()

# 2. Compute mean abeta_number per microglia_subtype
mean_abeta = (
    obs
    .groupby('microglia_subtype')['abeta_number']
    .mean()
    .reset_index(name='mean_abeta')
)

# 3. Print the table
print(mean_abeta)

# 4. Plot as a bar chart
plt.figure(figsize=(8, 4))
plt.bar(
    mean_abeta['microglia_subtype'],
    mean_abeta['mean_abeta']
)
plt.xticks(rotation=45, ha='right')
plt.ylabel('Average Aβ (abeta_number)')
plt.title('Mean Aβ per Microglia Subtype')
plt.tight_layout()
plt.show()

In [ ]:
# 1. Map samples to broader groups
group_map = {
    'Low_3_2':  '0.004',
    'Low_2_2':  '0.019',
    'Low_5_2':  '0.166',
    'Low_4_2':  '0.053',
    'Low_12_2': '0.014',
    'Low_13_2': '0.014',
    'Low_11_2': '0.01',
    'Low_10_2': '0.057',
    'Low_16_2': '0.119',
    'Low_31':   '0.088',
    'High_7_2':  '0.064',
    'High_1_2':  '0.111',
    'High_9_2':  '0.019',
    'High_19_2': '0.179',
    'High_33':   '0.42',
    'High_34':   '0.691',
    'Non_1':    '0.111',
    'Non_28':   '0.047',
    'Non_2':    '0.016',
    'Non_35':   '0.142',
    'Non_8_2':  '0.081',
    'Non_14_2': '0.041'
}
adata_micro.obs['abeta_number'] = adata.obs['sample'].map(group_map)#.astype('category')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Extract the obs DataFrame
obs = adata_micro.obs.copy()

# 2. Compute mean tau_number per micro_subtype
mean_tau = (
    obs
    .groupby('microglia_subtype')['tau_number']
    .mean()
    .reset_index(name='mean_tau')
)

# 3. Print the table
print(mean_tau)

# 4. Plot as a bar chart
plt.figure(figsize=(8, 4))
plt.bar(mean_tau['microglia_subtype'], mean_tau['mean_tau'])
plt.xticks(rotation=45, ha='right')
plt.ylabel('Average τ (tau_number)')
plt.title('Mean τ per Micro Subtype')
plt.tight_layout()
plt.show()

In [ ]:
# 2. Map and convert to float
adata_micro.obs['abeta_number'] = (
    adata_micro.obs['sample']
    .map(group_map)        # this yields floats or NaN
    .astype(float)         # ensure the dtype is numeric
)

# 3. (Optionally) check dtype and missing values
print(adata_micro.obs['abeta_number'].dtype)  # should be float64
print(adata_micro.obs['abeta_number'].isna().sum(), "missing mappings")

In [ ]:
ov.pl.embedding(adata_micro,basis='X_umap',
                   color=['tau_number','abeta_number','microglia_subtype','SORL1'],
                   frameon='small',cmap='Reds')

In [ ]:
adata_micro

In [ ]:
import numpy as np
import pandas as pd

# 1. Ensure your adata_micro has raw or normalized expression matrix
#    Here I’ll pull from .X (assumed log‐normalized)
expr = adata_micro.X.copy()            # shape (n_cells, n_genes)
genes = adata_micro.var_names           # array of gene names
tau = adata_micro.obs['tau_number'].values  # shape (n_cells,)

# 2. Center both expression and tau for correlation
expr_centered = expr - expr.mean(axis=0)
tau_centered = tau - tau.mean()

# 3. Compute Pearson correlation coefficients for each gene
#    corr_j = (expr_centered[:,j] · tau_centered) / (||expr_centered[:,j]|| * ||tau_centered||)
num = expr_centered.T.dot(tau_centered)                     # shape (n_genes,)
den = np.linalg.norm(expr_centered, axis=0) * np.linalg.norm(tau_centered)
corrs = num / den

# 4. Pick top 100 genes
top_idx = np.argsort(corrs)[-100:][::-1]   # indices of top 100
top_genes = genes[top_idx]
top_corrs = corrs[top_idx]

# 5. Build summary DataFrame
top100_df = pd.DataFrame({
    'gene': top_genes,
    'pearson_r': top_corrs
})

print(top100_df)

In [ ]:
adata_micro

In [ ]:
marker_genes = ['SORL1','tau_number','abeta_number']
sc.pl.dotplot(adata_micro, marker_genes, groupby='microglia_subtype',
             standard_scale='var');

In [ ]:
DEgene = pd.read_csv('./DE_results_new/DE_genes_all_celltypes.csv')
DEgene

In [ ]:
adata_micro

In [ ]:
import numpy as np
import pandas as pd

# 1. Pull out the expression matrix, converting sparse → dense if needed
expr = (
    adata_micro.X.toarray()
    if hasattr(adata_micro.X, "toarray")
    else adata_micro.X
)  # shape: (n_cells, n_genes)

# 2. Convert var_names to a 1D numpy array
genes_array = adata_micro.var_names.to_numpy()  # now a plain ndarray

# 3. Pull tau_number as a vector
tau = adata_micro.obs['tau_number'].values       # shape: (n_cells,)

# 4. Center both for Pearson correlation
expr_centered = expr - expr.mean(axis=0)
tau_centered = tau - tau.mean()

# 5. Compute gene‑wise Pearson r
numerator = expr_centered.T.dot(tau_centered)      # (n_genes,)
denominator = np.linalg.norm(expr_centered, axis=0) * np.linalg.norm(tau_centered)
corrs = numerator / denominator

# 6. Grab the top 100 positively correlated genes
top_idx = np.argsort(corrs)[-100:][::-1]           # indices of top 100
top_genes = genes_array[top_idx]                   # safe numpy indexing
top_corrs = corrs[top_idx]

# 7. Build a DataFrame for inspection
top100_df = pd.DataFrame({
    'gene': top_genes,
    'pearson_r': top_corrs
})

print(top100_df)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Step 0 (optional): if you haven't kept top100_df, recompute it ---
expr = adata_micro.X.toarray() if hasattr(adata_micro.X, "toarray") else adata_micro.X
genes = adata_micro.var_names.to_numpy()
tau = adata_micro.obs['tau_number'].values

# Center for Pearson
expr_centered = expr - expr.mean(axis=0)
tau_centered = tau - tau.mean()

num = expr_centered.T.dot(tau_centered)
den = np.linalg.norm(expr_centered, axis=0) * np.linalg.norm(tau_centered)
corrs = num / den

top_idx = np.argsort(corrs)[-100:][::-1]
top_genes = genes[top_idx]

# --- Step 1: extract and sort expression ---
expr_top = expr[:, top_idx]          # all cells × top100 genes
cell_order = np.argsort(tau)         # order of cells by tau_number
expr_sorted = expr_top[cell_order, :]

# --- Step 2: plot heatmap ---
plt.figure(figsize=(12, 10))
plt.imshow(expr_sorted.T, aspect='auto')
plt.colorbar(label='Expression')
plt.yticks(np.arange(len(top_genes)), top_genes, fontsize=6)
plt.xlabel('Cells (sorted by τ)')
plt.ylabel('Top 100 Correlated Genes')
plt.title('Heatmap of Top 100 Genes Correlated with τ')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec
import pandas as pd

# --- 1. Recompute / fetch necessary data --------------------------------------

# Expression: dense matrix (n_cells × n_genes)
expr = adata_micro.X.toarray() if hasattr(adata_micro.X, "toarray") else adata_micro.X
genes = adata_micro.var_names.to_numpy()
tau = adata_micro.obs['tau_number'].values

# Center for correlation
expr_centered = expr - expr.mean(axis=0)
tau_centered = tau - tau.mean()
num = expr_centered.T.dot(tau_centered)
den = np.linalg.norm(expr_centered, axis=0) * np.linalg.norm(tau_centered)
corrs = num / den

# Top 100 most positively correlated genes
top_idx = np.argsort(corrs)[-100:][::-1]
top_genes = genes[top_idx]
expr_top = expr[:, top_idx]

# Order cells by tau
cell_order = np.argsort(tau)
expr_sorted = expr_top[cell_order, :]

# Get subtype for each ordered cell
subtypes = adata_micro.obs['microglia_subtype'].values[cell_order]

# --- 2. Define a color for each subtype --------------------------------------

# Generate a distinct color for each subtype
unique_sub = pd.Categorical(adata_micro.obs['microglia_subtype']).categories
cmap = plt.get_cmap('tab20')
color_dict = {
    sub: cmap(i % cmap.N)
    for i, sub in enumerate(unique_sub)
}
# Map ordered subtypes to colors
sub_colors = [color_dict[s] for s in subtypes]

# --- 3. Plot with a color‐strip annotation ------------------------------------

fig = plt.figure(figsize=(14, 10))
gs = gridspec.GridSpec(2, 1, height_ratios=[0.3, 9], hspace=0.05)

# a) Top annotation bar
ax0 = fig.add_subplot(gs[0])
ax0.imshow([sub_colors], aspect='auto')
ax0.set_xticks([])
ax0.set_yticks([])
ax0.set_ylabel('Subtype', rotation=0, labelpad=40, va='center')

# b) Main heatmap
ax1 = fig.add_subplot(gs[1], sharex=ax0)
im = ax1.imshow(expr_sorted.T, aspect='auto')
ax1.set_yticks(np.arange(len(top_genes)))
ax1.set_yticklabels(top_genes, fontsize=6)
ax1.set_xlabel('Cells (sorted by τ)')
ax1.set_ylabel('Top 100 Correlated Genes')
ax1.set_title('Heatmap of Top 100 Genes Correlated with τ\n+ Microglia Subtype Annotation')
plt.setp(ax1.get_xticklabels(), visible=False)  # hide x‐tick labels for clarity

# c) Colorbar for expression values
cbar = fig.colorbar(im, ax=[ax1], orientation='vertical', fraction=0.02, pad=0.02)
cbar.set_label('Expression')

# d) Legend for subtypes
handles = [
    plt.Line2D([0], [0], marker='s', color=color_dict[sub], linestyle='') 
    for sub in unique_sub
]
fig.legend(handles, unique_sub, title='Microglia Subtype',
           loc='upper right', bbox_to_anchor=(0.95, 0.9))

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec
import pandas as pd

# --- 1. Recompute / fetch necessary data --------------------------------------

# Expression: dense matrix (n_cells × n_genes)
expr = adata_micro.X.toarray() if hasattr(adata_micro.X, "toarray") else adata_micro.X
genes = adata_micro.var_names.to_numpy()
tau = adata_micro.obs['tau_number'].values

# Center for correlation
expr_centered = expr - expr.mean(axis=0)
tau_centered = tau - tau.mean()
num = expr_centered.T.dot(tau_centered)
den = np.linalg.norm(expr_centered, axis=0) * np.linalg.norm(tau_centered)
corrs = num / den

# Top 50 most positively correlated genes
top_idx = np.argsort(corrs)[-50:][::-1]
top_genes = genes[top_idx]
expr_top = expr[:, top_idx]

# Order cells by tau
cell_order = np.argsort(tau)
expr_sorted = expr_top[cell_order, :]

# Get subtype for each ordered cell
subtypes = adata_micro.obs['microglia_subtype'].values[cell_order]

# --- 2. Define a color for each subtype --------------------------------------

unique_sub = pd.Categorical(adata_micro.obs['microglia_subtype']).categories
cmap_sub = plt.get_cmap('tab20')
color_dict = {sub: cmap_sub(i % cmap_sub.N) for i, sub in enumerate(unique_sub)}
sub_colors = [color_dict[s] for s in subtypes]

# --- 3. Plot with a color-strip annotation and side legend -------------------

fig = plt.figure(figsize=(14, 12))
gs = gridspec.GridSpec(2, 1, height_ratios=[0.3, 9], hspace=0.05)

# a) Top annotation bar
ax0 = fig.add_subplot(gs[0])
ax0.imshow([sub_colors], aspect='auto')
ax0.set_xticks([])
ax0.set_yticks([])
ax0.set_ylabel('Subtype', rotation=0, labelpad=40, va='center')

# b) Main heatmap
ax1 = fig.add_subplot(gs[1], sharex=ax0)
im = ax1.imshow(expr_sorted.T, aspect='auto', cmap='coolwarm')
ax1.set_yticks(np.arange(len(top_genes)))
ax1.set_yticklabels(top_genes, fontsize=6)
ax1.set_xlabel('Cells (sorted by τ)')
ax1.set_ylabel('Top 50 Correlated Genes')
ax1.set_title('Heatmap of Top 50 Genes Correlated with τ\n+ Microglia Subtype Annotation')
plt.setp(ax1.get_xticklabels(), visible=False)  # hide x–tick labels for clarity

# c) Colorbar for expression values (cyan low, red high)
cbar = fig.colorbar(im, ax=ax1, orientation='vertical', fraction=0.02, pad=0.02)
cbar.set_label('Expression')

# d) Legend for subtypes on the right side
handles = [plt.Line2D([0], [0], marker='s', color=color_dict[sub], linestyle='') for sub in unique_sub]
fig.legend(handles, unique_sub, title='Microglia Subtype',
           loc='center right', bbox_to_anchor=(1.15, 0.5), frameon=False)

plt.tight_layout(rect=[0, 0, 0.85, 1])  # leave space for legend
plt.show()


In [ ]:
genes = genes.astype(str)  # 显式转换为字符串
non_rp_mask = ~np.char.startswith(genes, 'RP')
non_rp_mask = ~np.char.startswith(genes, 'RP')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec
import pandas as pd

# --- 1. Recompute / fetch necessary data --------------------------------------

# Expression matrix: n_cells × n_genes
expr = adata_micro.X.toarray() if hasattr(adata_micro.X, "toarray") else adata_micro.X
genes = adata_micro.var_names.to_numpy().astype(str)  # Ensure string type
tau = adata_micro.obs['tau_number'].values

# --- 1a. Remove ribosomal genes ("RP") and Ensembl IDs ("ENSG") ---------------
non_rp_mask = ~np.char.startswith(genes, 'RP')
non_ensg_mask = ~np.char.find(genes, 'ENSG') >= 0
valid_gene_mask = non_rp_mask & non_ensg_mask

expr = expr[:, valid_gene_mask]
genes = genes[valid_gene_mask]

# --- 1b. Compute gene–tau correlations ----------------------------------------

# Center the data for correlation computation
expr_centered = expr - expr.mean(axis=0)
tau_centered = tau - tau.mean()

# Dot product for correlation
num = expr_centered.T.dot(tau_centered)
den = np.linalg.norm(expr_centered, axis=0) * np.linalg.norm(tau_centered)
corrs = num / den

# Top 50 most positively correlated genes
top_idx = np.argsort(corrs)[-50:][::-1]
top_genes = genes[top_idx]
expr_top = expr[:, top_idx]

# Order cells by tau
cell_order = np.argsort(tau)
expr_sorted = expr_top[cell_order, :]

# Get subtype for each ordered cell
subtypes = adata_micro.obs['microglia_subtype'].values[cell_order]

# --- 2. Define a color for each subtype --------------------------------------

unique_sub = pd.Categorical(adata_micro.obs['microglia_subtype']).categories
cmap_sub = plt.get_cmap('tab20')
color_dict = {sub: cmap_sub(i % cmap_sub.N) for i, sub in enumerate(unique_sub)}
sub_colors = [color_dict[s] for s in subtypes]

# --- 3. Plot with color-strip annotation and side legend ----------------------

fig = plt.figure(figsize=(14, 12))
gs = gridspec.GridSpec(2, 1, height_ratios=[0.3, 9], hspace=0.05)

# a) Top annotation bar
ax0 = fig.add_subplot(gs[0])
ax0.imshow([sub_colors], aspect='auto')
ax0.set_xticks([])
ax0.set_yticks([])
ax0.set_ylabel('Subtype', rotation=0, labelpad=40, va='center')

# b) Main heatmap
ax1 = fig.add_subplot(gs[1], sharex=ax0)
im = ax1.imshow(expr_sorted.T, aspect='auto', cmap='coolwarm')
ax1.set_yticks(np.arange(len(top_genes)))
ax1.set_yticklabels(top_genes, fontsize=6)
ax1.set_xlabel('Cells (sorted by τ)')
ax1.set_ylabel('Top 50 Correlated Genes')
ax1.set_title('Heatmap of Top 50 Genes Correlated with τ\n+ Microglia Subtype Annotation')
plt.setp(ax1.get_xticklabels(), visible=False)  # hide x–tick labels for clarity

# c) Colorbar for expression values
cbar = fig.colorbar(im, ax=ax1, orientation='vertical', fraction=0.02, pad=0.02)
cbar.set_label('Expression')

# d) Legend for subtypes on the right
handles = [plt.Line2D([0], [0], marker='s', color=color_dict[sub], linestyle='') for sub in unique_sub]
fig.legend(handles, unique_sub, title='Microglia Subtype',
           loc='center right', bbox_to_anchor=(1.15, 0.5), frameon=False)

plt.tight_layout(rect=[0, 0, 0.85, 1])  # leave space for legend
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.colors import LinearSegmentedColormap
import pandas as pd

# --- Data preparation (unchanged) ---------------------------------------------

expr = adata_micro.X.toarray() if hasattr(adata_micro.X, "toarray") else adata_micro.X
genes = adata_micro.var_names.to_numpy().astype(str)
tau = adata_micro.obs['tau_number'].values

non_rp_mask = ~np.char.startswith(genes, 'RP')
non_ensg_mask = ~np.char.find(genes, 'ENSG') >= 0
valid_gene_mask = non_rp_mask & non_ensg_mask

expr = expr[:, valid_gene_mask]
genes = genes[valid_gene_mask]

expr_centered = expr - expr.mean(axis=0)
tau_centered = tau - tau.mean()
num = expr_centered.T.dot(tau_centered)
den = np.linalg.norm(expr_centered, axis=0) * np.linalg.norm(tau_centered)
corrs = num / den

top_idx = np.argsort(corrs)[-50:][::-1]
top_genes = genes[top_idx]
expr_top = expr[:, top_idx]

cell_order = np.argsort(tau)
expr_sorted = expr_top[cell_order, :]
subtypes = adata_micro.obs['microglia_subtype'].values[cell_order]

# --- Subtype colors -----------------------------------------------------------

unique_sub = pd.Categorical(adata_micro.obs['microglia_subtype']).categories
cmap_sub = plt.get_cmap('tab20')
color_dict = {sub: cmap_sub(i % cmap_sub.N) for i, sub in enumerate(unique_sub)}
sub_colors = [color_dict[s] for s in subtypes]

# --- Custom red–cyan colormap -------------------------------------------------

red_cyan_cmap = LinearSegmentedColormap.from_list("red_cyan", ["cyan", "white", "red"])

# --- Plotting: aligned subtype bar + heatmap ----------------------------------

fig = plt.figure(figsize=(14, 12))
gs = gridspec.GridSpec(2, 1, height_ratios=[0.3, 9], hspace=0.05)

# a) Subtype annotation bar (top)
ax0 = fig.add_subplot(gs[0])
ax0.imshow([sub_colors], aspect='auto', extent=[0, expr_sorted.shape[0], 0, 1])
ax0.set_xlim([0, expr_sorted.shape[0]])
ax0.set_xticks([])
ax0.set_yticks([])
ax0.set_ylabel('Subtype', rotation=0, labelpad=40, va='center')

# b) Main heatmap (bottom)
ax1 = fig.add_subplot(gs[1], sharex=ax0)
im = ax1.imshow(expr_sorted.T, aspect='auto', cmap=red_cyan_cmap, extent=[0, expr_sorted.shape[0], 0, len(top_genes)])
ax1.set_xlim([0, expr_sorted.shape[0]])
ax1.set_yticks(np.arange(len(top_genes)) + 0.5)
ax1.set_yticklabels(top_genes, fontsize=6)
ax1.set_xlabel('Cells (sorted by τ)')
ax1.set_ylabel('Top 50 Correlated Genes')
ax1.set_title('Heatmap of Top 50 Genes Correlated with τ\n+ Microglia Subtype Annotation')
plt.setp(ax1.get_xticklabels(), visible=False)

# c) Colorbar
cbar = fig.colorbar(im, ax=ax1, orientation='vertical', fraction=0.02, pad=0.02)
cbar.set_label('Expression')

# d) Legend
handles = [plt.Line2D([0], [0], marker='s', color=color_dict[sub], linestyle='') for sub in unique_sub]
fig.legend(handles, unique_sub, title='Microglia Subtype',
           loc='center right', bbox_to_anchor=(1.15, 0.5), frameon=False)

# Layout and save
plt.tight_layout(rect=[0, 0, 0.85, 1])
fig.savefig("./output/microglia_tau_heatmap_aligned_20250609.tif", format='tiff', dpi=300)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec
import pandas as pd

# --- 1. Recompute / fetch necessary data --------------------------------------

# Expression: dense matrix (n_cells × n_genes)
expr = adata_micro.X.toarray() if hasattr(adata_micro.X, "toarray") else adata_micro.X
genes = adata_micro.var_names.to_numpy()
tau = adata_micro.obs['tau_number'].values

# Filter out ribosomal genes (those starting with "RP")
non_rp_mask = ~np.char.startswith(genes, 'RP')
expr = expr[:, non_rp_mask]
genes = genes[non_rp_mask]

# Center for correlation
expr_centered = expr - expr.mean(axis=0)
tau_centered = tau - tau.mean()
num = expr_centered.T.dot(tau_centered)
den = np.linalg.norm(expr_centered, axis=0) * np.linalg.norm(tau_centered)
corrs = num / den

# Top 50 most positively correlated genes
top_idx = np.argsort(corrs)[-50:][::-1]
top_genes = genes[top_idx]
expr_top = expr[:, top_idx]

# Order cells by tau
cell_order = np.argsort(tau)
expr_sorted = expr_top[cell_order, :]

# Get subtype for each ordered cell
subtypes = adata_micro.obs['microglia_subtype'].values[cell_order]

# --- 2. Define a color for each subtype --------------------------------------

unique_sub = pd.Categorical(adata_micro.obs['microglia_subtype']).categories
cmap_sub = plt.get_cmap('tab20')
color_dict = {sub: cmap_sub(i % cmap_sub.N) for i, sub in enumerate(unique_sub)}
sub_colors = [color_dict[s] for s in subtypes]

# --- 3. Plot with a color-strip annotation and side legend -------------------

fig = plt.figure(figsize=(8, 8))
gs = gridspec.GridSpec(2, 1, height_ratios=[0.3, 9], hspace=0.05)

# a) Top annotation bar
ax0 = fig.add_subplot(gs[0])
ax0.imshow([sub_colors], aspect='auto')
ax0.set_xticks([])
ax0.set_yticks([])
ax0.set_ylabel('Subtype', rotation=0, labelpad=40, va='center')

# b) Main heatmap
ax1 = fig.add_subplot(gs[1], sharex=ax0)
im = ax1.imshow(expr_sorted.T, aspect='auto', cmap='coolwarm')
ax1.set_yticks(np.arange(len(top_genes)))
ax1.set_yticklabels(top_genes, fontsize=6)
ax1.set_xlabel('Cells (sorted by τ)')
ax1.set_ylabel('Top 50 Correlated Genes')
ax1.set_title('Heatmap of Top 50 Genes Correlated with τ\n+ Microglia Subtype Annotation')
plt.setp(ax1.get_xticklabels(), visible=False)

# c) Colorbar for expression values
cbar = fig.colorbar(im, ax=ax1, orientation='vertical', fraction=0.02, pad=0.02)
cbar.set_label('Expression')

# d) Legend for subtypes
handles = [plt.Line2D([0], [0], marker='s', color=color_dict[sub], linestyle='') for sub in unique_sub]
fig.legend(handles, unique_sub, title='Microglia Subtype',
           loc='center right', bbox_to_anchor=(1.15, 0.5), frameon=False)

plt.tight_layout(rect=[0, 0, 0.85, 1])
plt.show()

In [ ]:
adata_micro.write_h5ad('./zhang/microglia_adata_20250520.h5ad')

In [ ]:
adata_micro = sc.read_h5ad('./zhang/microglia_adata_20250520.h5ad')

In [ ]:
adata_micro

In [ ]:
import numpy as np
import pandas as pd

# 1. Extract expression matrix and gene names
expr = adata_micro.X.toarray() if hasattr(adata_micro.X, "toarray") else adata_micro.X
genes = adata_micro.var_names.to_numpy()
tau = adata_micro.obs['tau_number'].values

# 2. Center the data for Pearson correlation
expr_centered = expr - expr.mean(axis=0)
tau_centered = tau - tau.mean()

# 3. Compute Pearson correlation coefficients
numerator = expr_centered.T @ tau_centered
denominator = np.linalg.norm(expr_centered, axis=0) * np.linalg.norm(tau_centered)
correlations = numerator / denominator

# 4. Identify top 50 positively correlated genes
top_indices = np.argsort(correlations)[-50:][::-1]
top_genes = genes[top_indices]

# 5. Extract expression data for top genes
expr_top = expr[:, top_indices]

# 6. Create a DataFrame with expression data and subtype information
df_expr = pd.DataFrame(expr_top, columns=top_genes)
df_expr['microglia_subtype'] = adata_micro.obs['microglia_subtype'].values

# 7. Group by subtype and calculate mean expression
mean_expression = df_expr.groupby('microglia_subtype').mean()

# 8. Reset index for better display
mean_expression.reset_index(inplace=True)

# 9. Display the result
print("Mean expression of top 50 tau-correlated genes by microglia_subtype:")
print(mean_expression)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming 'mean_expression' is the DataFrame with mean gene expressions
# Set 'microglia_subtype' as the index
mean_expression.set_index('microglia_subtype', inplace=True)

# Create a heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(mean_expression, cmap='coolwarm', annot=False)

# Customize the plot
plt.title('Mean Tau related Gene Expression per Microglia Subtype')
plt.xlabel('Genes')
plt.ylabel('Microglia Subtypes')
plt.tight_layout()

# Display the heatmap
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Extract full expression matrix and gene names
expr = adata_micro.X.toarray() if hasattr(adata_micro.X, "toarray") else adata_micro.X
subtypes = adata_micro.obs['microglia_subtype'].values

# Create DataFrame with all genes
df_all_expr = pd.DataFrame(expr)
df_all_expr['microglia_subtype'] = subtypes

# Group by subtype and calculate mean expression across all genes
mean_expression_all = df_all_expr.groupby('microglia_subtype').mean()
mean_expression_all['mean_expression_all_gene'] = mean_expression_all.mean(axis=1)

# Reset index for plotting
mean_expression_all.reset_index(inplace=True)

# Plotting the mean expression of all genes
plt.figure(figsize=(12, 6))
plt.bar(mean_expression_all['microglia_subtype'], mean_expression_all['mean_expression_all_gene'])
plt.xticks(rotation=45, ha='right', fontsize=10, fontweight='bold')
plt.ylabel("Mean Expression (All Genes)", fontsize=12)
plt.title("Mean Expression of Tau related Genes per Microglia Subtype", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
adata

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize = (1.5,4))
ov.pl.cellproportion(adata=adata,celltype_clusters='class',
                    groupby='Tau',legend=True,ax=ax)

In [ ]:
adata

In [ ]:
%%time
adata=ov.pp.preprocess(adata,mode='shiftlog|pearson',n_HVGs=2000,
                       target_sum=50*1e4)
adata

In [ ]:
%%time
ov.pp.scale(adata)
adata

In [ ]:
%%time
ov.pp.pca(adata,layer='scaled',n_pcs=50)
adata

In [ ]:
sc.tl.dendrogram(adata,'class',use_rep='X_pca')
sc.tl.rank_genes_groups(adata, 'class', use_rep='X_pca',
                        method='class',use_raw=False,key_added='class_ttest')
sc.pl.rank_genes_groups_dotplot(adata,groupby='class',
                                cmap='Spectral_r',key='class_ttest',
                                standard_scale='var',n_genes=5)

In [ ]:
import pandas as pd

# Assume DEgene is a DataFrame with columns:
#   'cell_type', 'names', 'logfoldchanges', 'pvals_adj'

df = DEgene.copy()

# Set significance thresholds
lfc_cutoff = 0.58         # log₂ fold-change threshold
pval_cutoff = 0.05        # adjusted p-value threshold

# Condition A: upregulated genes
condA = df[
    (df['logfoldchanges'] > lfc_cutoff) &
    (df['pvals'] < pval_cutoff)
]
counts_up = (
    condA
    .groupby('cell_type')['names']
    .nunique()
    .rename('count_up')
)

# Condition B: downregulated genes
condB = df[
    (df['logfoldchanges'] < -lfc_cutoff) &
    (df['pvals'] < pval_cutoff)
]
counts_down = (
    condB
    .groupby('cell_type')['names']
    .nunique()
    .rename('count_down')
)

# Combine into one summary table
summary = (
    pd.concat([counts_up, counts_down], axis=1)
      .fillna(0)
      .astype(int)
      .reset_index()
)

print(summary)

In [ ]:
# 比如 adata.obs['group'] 中存储了每个细胞的分组标签："AD" 或 "Non_AD"
adata_AD = adata[adata.obs['AD'] == 'AD', :].copy()
adata_non_AD = adata[adata.obs['AD'] == 'Non_AD', :].copy()

In [ ]:
import numpy as np
import omicverse as ov
# restore the old alias so pygam’s calls to np.int still work
np.int = int
# now import VIA and pygam as usual
from omicverse.externel.VIA import core
import pygam as pg
import scanpy as sc
import omicverse as ov
from omicverse.externel import VIA

import matplotlib.pyplot as plt
ov.plot_set()

In [ ]:
import omicverse as ov
#import VIA
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt

# 1. Parameters
ncomps            = 30
knn               = 15
v0_random_seed    = 4
# Pick the representation you actually have:
use_rep           = 'X_pca'                     
clusters          = 'microglia_subtype'
basis             = 'X_umap'
dataset           = ''         # only 'group' if you select a cell type root
memory            = 10

# 2. Choose a single root cell by its index or obs_name:
#    e.g. pick the first cell of subtype "Redox(SORL1 low)":
candidate_cells = adata_non_AD.obs_names[adata_non_AD.obs[clusters] == 'Redox(SORL1 low)']
if len(candidate_cells) == 0:
    raise ValueError("No cells found in subtype 'Redox(SORL1 low)'")
root_user = [candidate_cells[0]]  

# 3. Extract the data matrix for VIA
X = adata_non_AD.obsm[use_rep][:, :ncomps]  # shape = (n_cells, ncomps)

# 4. Run VIA
v0 = VIA.core.VIA(
    data=X,
    true_label=adata_non_AD.obs[clusters].values,
    knn=knn,
    root_user=root_user,
    resolution_parameter=1.5,
    edgepruning_clustering_resolution=0.15,
    cluster_graph_pruning=0.15,
    dataset=dataset,
    random_seed=v0_random_seed,
    memory=memory
)
v0.run_VIA()

# 5. Pie‑chart graph
fig, ax, ax1 = VIA.core.plot_piechart_viagraph_ov(
    adata_non_AD,
    clusters=clusters,
    dpi=300,
    via_object=v0,
    ax_text=False,
    show_legend=False
)
fig.set_size_inches(8, 4)

# 6. Trajectory curves (only once)
fig, ax, ax1 = VIA.core.plot_trajectory_curves_ov(
    adata_non_AD,
    clusters=clusters,
    dpi=300,
    via_object=v0,
    embedding=adata_non_AD.obsm[basis],
    draw_all_curves=False
)

# 7. Atlas view colored by pseudotime
v0.embedding = adata_AD.obsm[basis]
fig, ax = VIA.core.plot_atlas_view(
    via_object=v0, 
    n_milestones=150, 
    sc_labels=adata_non_AD.obs[clusters],
    fontsize_title=12,
    fontsize_labels=12,
    dpi=300,
    extra_title_text='Atlas View colored by pseudotime'
)
fig.set_size_inches(4, 4)

# 8. Streamplot
fig, ax = VIA.core.via_streamplot_ov(
    adata_non_AD,
    clusters,
    v0,
    embedding=adata_non_AD.obsm[basis],
    dpi=300,
    density_grid=0.8,
    scatter_size=30,
    scatter_alpha=0.3,
    linewidth=0.5
)
fig.set_size_inches(5, 5)


In [ ]:
# 0. Imports
import omicverse as ov
#import VIA
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt

# 1. Parameters
ncomps         = 30
knn            = 15
v0_random_seed = 4
use_rep        = 'X_pca'             # one valid key in adata.obsm
clusters       = 'microglia_subtype'
basis          = 'X_umap'
dataset        = ''                  # only 'group' if using cell‑type as root
memory         = 10

# 2. Select a single root cell from adata_non_AD
candidate_cells = adata_AD.obs_names[adata_AD.obs[clusters] == 'Redox(SORL1 low)']
if len(candidate_cells) == 0:
    raise ValueError("No cells found in subtype 'Redox(SORL1 low)'")
root_user = [candidate_cells[0]]

# 3. Extract the VIA input matrix
X = adata_AD.obsm[use_rep][:, :ncomps]   # shape = (n_cells, ncomps)

# 4. Initialize and run VIA
v0 = VIA.core.VIA(
    data=X,
    true_label=adata_AD.obs[clusters].values,
    knn=knn,
    root_user=root_user,
    resolution_parameter=1.5,
    edgepruning_clustering_resolution=0.15,
    cluster_graph_pruning=0.15,
    dataset=dataset,
    random_seed=v0_random_seed,
    memory=memory
)
v0.run_VIA()

# 5. Pie‑chart of the VIA graph
fig, ax, ax1 = VIA.core.plot_piechart_viagraph_ov(
    adata_AD,
    clusters=clusters,
    dpi=300,
    via_object=v0,
    ax_text=False,
    show_legend=False
)
fig.set_size_inches(8, 4)

# 6. Trajectory curves on UMAP
fig, ax, ax1 = VIA.core.plot_trajectory_curves_ov(
    adata_AD,
    clusters=clusters,
    dpi=300,
    via_object=v0,
    embedding=adata_AD.obsm[basis],
    draw_all_curves=False
)

# 7. Atlas view colored by pseudotime
#    → Must use the same AnnData for embedding and labels
v0.embedding = adata_AD.obsm[basis]
fig, ax = VIA.core.plot_atlas_view(
    via_object=v0,
    n_milestones=150,
    sc_labels=adata_AD.obs[clusters],
    fontsize_title=12,
    fontsize_labels=12,
    dpi=300,
    extra_title_text='Atlas View colored by pseudotime'
)
fig.set_size_inches(4, 4)

# 8. Streamplot on UMAP
fig, ax = VIA.core.via_streamplot_ov(
    adata_AD,
    clusters,
    v0,
    embedding=adata_AD.obsm[basis],
    dpi=300,
    density_grid=0.8,
    scatter_size=30,
    scatter_alpha=0.3,
    linewidth=0.5
)
fig.set_size_inches(5, 5)

In [ ]:
adata.write_h5ad('./zhang/microglia_final_20250520.h5ad')

In [ ]:
adata_micro = sc.read_h5ad('./zhang/microglia_final_20250520.h5ad')

In [ ]:
adata_micro

In [ ]:
ov.pl.embedding(adata_micro,basis='X_umap',
                   color=['microglia_subtype','SORL1'],
                   frameon='small',cmap='Reds')

In [ ]:
adata_ac = sc.read_h5ad('./zhang/adata_astrocyte_20250519.h5ad')

In [ ]:
adata_ac

In [ ]:
marker_genes = ['SLC1A2','ARHGAP24','SLC1A3']
sc.pl.dotplot(adata_ac, marker_genes, groupby='astrocyte_subtype',standard_scale='var');

In [ ]:
adata_micro